# Pair Inputs by Name

Pair two independently generated streams by filename stem. The installed `CsvPair` example keeps both byte streams and declares their exact input positions. A per-step grouping policy determines which pairs reach it.

In [ ]:
from __future__ import annotations

from artisan.inspection import inspect_evidence
from artisan.operations.curator import Merge
from artisan.operations.examples import CsvPair, DataGenerator
from artisan.orchestration import PipelineManager, StepStatus
from artisan.schemas import GroupByStrategy
from artisan.storage import ArtifactStore
from artisan.utils import tutorial_setup
from artisan.visualization import build_macro_graph, build_micro_graph


def data_at_step(env, run_id, step_number):
    refs = inspect_evidence(
        "query_artifacts",
        {"pipeline_run_id": run_id, "artifact_type": "csv", "limit": 100},
        delta_root=env.delta_root,
    )
    ids = [
        item.artifact_id
        for item in refs.items
        if item.current_step_number == step_number
    ]
    return ArtifactStore(env.delta_root).get_artifacts_by_type(ids, "csv")

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)

In [ ]:
env = tutorial_setup("name_basic")
with PipelineManager.create(
    name="name_basic",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
) as pipeline:
    pipeline.run(DataGenerator, name="left", params={"count": 2, "seed": 42})
    pipeline.run(DataGenerator, name="right", params={"count": 2, "seed": 99})
    paired = pipeline.run(
        CsvPair,
        name="pair",
        group_by=GroupByStrategy.NAME,
        inputs={
            "left": pipeline.output("left", "datasets"),
            "right": pipeline.output("right", "datasets"),
        },
        batch_strategy={"artifacts_per_unit": 2},
    )
    summary = pipeline.finalize()
assert summary["overall_success"], summary
left = data_at_step(env, pipeline.config.pipeline_run_id, 0)
right = data_at_step(env, pipeline.config.pipeline_run_id, 1)
right_by_name = {a.original_name: a for a in right.values()}
expected = {
    a.content + b"\n---\n" + right_by_name[a.original_name].content
    for a in left.values()
}
actual = data_at_step(env, pipeline.config.pipeline_run_id, paired.step_number)
assert len(actual) == 2
assert {a.content for a in actual.values()} == expected

The public provenance view includes both declared roles. Inspection reads accepted graph evidence rather than raw storage tables.

In [ ]:
edges = ArtifactStore(env.delta_root).provenance.load_edges_df(
    0, paired.step_number, include_roles=True
)
parents = edges.filter(edges["source_role"].is_in(["left", "right"]))
assert parents.height == 4
assert set(parents["target_artifact_id"]) == set(actual)
parents.select("source_artifact_id", "target_artifact_id", "source_role")

In [ ]:
build_macro_graph(env.delta_root)

In [ ]:
build_micro_graph(env.delta_root)

## Unmatched names

An unmatched stem produces no pair. Changing the transport batch size does not change the two matched outputs.

In [ ]:
other = tutorial_setup("name_unmatched")
with PipelineManager.create(
    name="name_unmatched",
    delta_root=other.delta_root,
    staging_root=other.staging_root,
    working_root=other.working_root,
) as unmatched:
    unmatched.run(DataGenerator, name="left", params={"count": 2, "seed": 42})
    unmatched.run(DataGenerator, name="right", params={"count": 3, "seed": 99})
    result = unmatched.run(
        CsvPair,
        group_by=GroupByStrategy.NAME,
        inputs={
            "left": unmatched.output("left", "datasets"),
            "right": unmatched.output("right", "datasets"),
        },
    )
    assert unmatched.finalize()["overall_success"]
assert {
    a.content
    for a in data_at_step(
        other, unmatched.config.pipeline_run_id, result.step_number
    ).values()
} == expected

## Duplicate names

Two different artifacts with the same stem make name pairing ambiguous. The step fails before a scientific unit is admitted; it has no outputs.

In [ ]:
duplicate_env = tutorial_setup("name_duplicates")
with PipelineManager.create(
    name="name_duplicates",
    delta_root=duplicate_env.delta_root,
    staging_root=duplicate_env.staging_root,
    working_root=duplicate_env.working_root,
) as duplicate:
    duplicate.run(DataGenerator, name="left_a", params={"count": 2, "seed": 42})
    duplicate.run(DataGenerator, name="left_b", params={"count": 2, "seed": 100})
    duplicate.run(
        Merge,
        name="left",
        inputs={
            "a": duplicate.output("left_a", "datasets"),
            "b": duplicate.output("left_b", "datasets"),
        },
    )
    duplicate.run(DataGenerator, name="right", params={"count": 2, "seed": 7})
    failed = duplicate.run(
        CsvPair,
        group_by=GroupByStrategy.NAME,
        inputs={
            "left": duplicate.output("left", "merged"),
            "right": duplicate.output("right", "datasets"),
        },
    )
    summary = duplicate.finalize()
assert failed.status is StepStatus.FAILED
assert "duplicate" in failed.error.lower()
assert failed.total_count == 0
assert not failed.output_roles
assert not summary["overall_success"]
assert (
    data_at_step(duplicate_env, duplicate.config.pipeline_run_id, failed.step_number)
    == {}
)